# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jh-emon002/flyrank-intern/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Provisional lane: Lane 2 — Refresh / Content Opportunity Scoring**.

I chose this lane because the practical decision is not simply whether a page is performing well or poorly, but which pages deserve limited human review time first. The starter dataset contains page-level search, content, freshness, visibility, and engagement signals that may help prioritize review candidates. My initial goal is therefore to investigate whether these observable signals can support a ranked review queue for content specialists. I will treat this lane as provisional and refine the target and validation strategy after examining the richer time-based data.

## 2. The question: decision, action, cost of a wrong call

**Research question**:
Can observable search, content, freshness, and engagement signals help prioritize which existing content items should be reviewed first for refresh or monitoring?

**Unit of analysis**:
One pseudonymized content item/page.

**Decision**:
Which pages should receive human review first?

**Output**:
A ranked review-priority score/queue.

**Action**:
An SEO specialist or editor examines the highest-priority pages and decides whether to refresh, expand, protect, monitor, or take no action.

**False-positive cost**:
Reviewer/editor time is spent on a page that did not need attention.

**False-negative cost**:
A potentially important deterioration/opportunity receives lower priority and may be missed.

**Why data/ML may help**:
Many content, visibility, freshness, position, and engagement signals may interact. ML may improve prioritization over a simple rule, but only if validation demonstrates meaningful improvement.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [9]:
from pathlib import Path
import pandas as pd
import subprocess

repo = Path("/content/flyrank-intern")

if not (repo / "data/raw/content_refresh_anonymized.csv").exists():
    subprocess.run(
        [
            "git", "clone", "-q",
            "https://github.com/jh-emon002/flyrank-intern.git",
            str(repo)
        ],
        check=True
    )

data_path = repo / "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [10]:
declining = df["trend_direction"].eq("down")

declining_with_demand = (
    declining &
    df["impressions_90d"].ge(100)
)

stale_visible = (
    df["days_since_last_update"].ge(180) &
    df["impressions_90d"].ge(500)
)

low_ctr_visible = (
    df["impressions_90d"].ge(500) &
    df["avg_position"].between(1, 20) &
    df["ctr"].lt(0.5)
)

print(f"Total content items: {len(df):,}")

print(
    f"Currently declining: {declining.sum():,} "
    f"({declining.mean():.1%})"
)

print(
    f"Declining with >=100 impressions: "
    f"{declining_with_demand.sum():,}"
)



Total content items: 30,000
Currently declining: 16,262 (54.2%)
Declining with >=100 impressions: 13,152


In [11]:
pct_declining_demand_all = declining_with_demand.mean() * 100
pct_declining_with_demand = (
    declining_with_demand.sum() / declining.sum() * 100
)
pct_low_ctr_visible = low_ctr_visible.mean() * 100

print(
    f"Declining + >=100 impressions: "
    f"{declining_with_demand.sum():,} "
    f"({pct_declining_demand_all:.1f}% of all items)"
)

print(
    f"Among declining items, {pct_declining_with_demand:.1f}% "
    f"have >=100 impressions"
)

print(
    f"Visible pages meeting provisional low-CTR rule: "
    f"{low_ctr_visible.sum():,} "
    f"({pct_low_ctr_visible:.1f}% of all items)"
)

Declining + >=100 impressions: 13,152 (43.8% of all items)
Among declining items, 80.9% have >=100 impressions
Visible pages meeting provisional low-CTR rule: 9,745 (32.5% of all items)


### interpretetion

The starter dataset contains 30,000 content items. Using the starter proxy, 16,262 items (54.2%) are currently classified as declining. Of these, 13,152 also have at least 100 impressions, meaning that approximately 80.9% of the currently declining items still have measurable search demand. This represents 43.8% of the full dataset.

These numbers suggest that decline alone is not sufficiently selective for deciding what a content team should review first. A team would still face more than 13,000 declining items with meaningful impressions. This supports framing the problem as prioritization: identifying which pages deserve limited human review time first, rather than simply predicting whether a page is declining.

## 4. Careful words: what I can and can't claim

This project may identify observed associations between page-level search/content signals and review-worthy performance patterns and may support prioritization decisions. It will not establish that changing a particular feature causes rankings or traffic to improve. A high-priority recommendation should therefore be treated as a candidate for human review rather than an automatic editing decision. I will avoid claims about “predicting Google's algorithm” and describe results as observed, directional, or decision-support unless a stronger study design justifies more.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.